# SRL paper figures

This notebook makes every figure in `pinatubo_srl.tex` and writes it to `figures/`. Nothing in `figures/` is edited by hand: rerun this notebook after the conversion (`10_conversion`) and analysis (`20_analysis`) notebooks, then rebuild the PDF with `latexmk -pdf pinatubo_srl.tex`.

| Figure | Label | Made here from | Output |
|---|---|---|---|
| 1 | `fig:network` | step 70 `maps(notes=False)` | `fig_network_early.png`, `fig_network_later.png` |
| 2 | `fig:pvo` | step 70 `diagrams(notes=False)` | `fig_pvo_system.png` |
| 3 | `fig:bob` | copy of step 69 output | `fig_bob_coverage.png` |
| 4 | `fig:clockrules` | `clock_regimes.csv` | `fig_clock_rules` |
| 5 | `fig:timing` | step 10 waveform index, step 43 origin index | `fig_timing_effect` |
| 6 | `fig:drift` | copy of step 69 output | `fig_ssam_rsam_lag.png` |
| 7 | `fig:anchors` | step 72 `plot_records` and its matches | `fig_clock_anchors` |
| 8 | `fig:workflow` | TikZ in the `.tex` file | — |
| 9 | `fig:stations` | step 10 station availability matrix | `fig_station_availability` |
| 10 | `fig:hypo` | step 43 event index | `fig_hypocentres` |
| 11 | `fig:daily` | steps 10, 22 and 43 | `fig_daily_counts` |
| 12 | `fig:june` | analysis run window metrics, step 43, BOB RSAM and SSAM | `fig_june_frequency` |

Captions live in the `.tex` file; caveats that the pipeline prints on its own figures (for example the map footnotes) are moved there. Step 70 and 72 functions are loaded from the cells tagged `pipeline_definitions` in those notebooks, the same way the tests load them, so the paper uses the pipeline's own plotting code.

## Configuration

In [ ]:
from pathlib import Path
import json, shutil, sys, tempfile

import matplotlib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PAPER = next(p for p in [Path.cwd(), Path.cwd() / 'papers/SRL'] if (p / 'pinatubo_srl.tex').is_file())
REPO = PAPER.parents[1]
PIPELINE = REPO / '10_conversion'
OUT = PAPER / 'figures'

sys.path.insert(0, str(PIPELINE))
from lib.config import pipeline_paths
P = pipeline_paths()

UTC = Path(P['OUTPUT_ROOT'])            # conversion products (10_conversion/conversion_config.json)
TMP = Path(P['TMP_DIR'])
BOB = Path(P['FAIR_TOP']) / 'BOB'
ANALYSIS_RUN = max((UTC.parent / 'analysis/notebook_workflow').glob('*/window_metrics.csv'),
                   key=lambda f: f.stat().st_mtime).parent   # newest analysis run

STYLE = {'font.size': 8, 'font.family': 'sans-serif', 'axes.spines.top': False, 'axes.spines.right': False}
plt.rcParams.update(STYLE)


def utc(s):
    # Parse UTC strings to naive UTC timestamps for plotting.
    return pd.to_datetime(s, utc=True, format='ISO8601').dt.tz_convert(None)


def save(fig, name, dpi=200, pdf=True):
    if pdf:
        fig.savefig(OUT / f'{name}.pdf', metadata={'CreationDate': None})   # reproducible bytes
    fig.savefig(OUT / f'{name}.png', dpi=dpi)
    plt.close(fig)
    print('wrote', name)


def pipeline_definitions(notebook, **globals_):
    # Execute the cells tagged pipeline_definitions in a pipeline notebook.
    # The notebooks import these in their setup cells, which are not tagged.
    from datetime import datetime, timezone
    ns = {'PIPELINE': PIPELINE, '__name__': 'paper_figures', 'np': np, 'pd': pd, 'plt': plt,
          'json': json, 'Path': Path, 'sys': sys, 'datetime': datetime, 'timezone': timezone, **globals_}
    cells = json.loads((PIPELINE / notebook).read_text())['cells']
    for c in cells:
        if 'pipeline_definitions' in c.get('metadata', {}).get('tags', []):
            exec(''.join(c['source']), ns)
    return ns

print('paper:', PAPER)
print('products:', UTC)
print('analysis run:', ANALYSIS_RUN.name, ANALYSIS_RUN.is_dir())

## Figures 1 and 2: network maps and PVO system (step 70)

`maps()` and `diagrams()` from step 70 with `notes=False`: no title or footnote, larger text. The footnote caveats are in the captions.

In [ ]:
ns70 = pipeline_definitions('70_network_history_and_stationxml.ipynb')
with tempfile.TemporaryDirectory() as d:
    d = Path(d)
    ns70['maps'](d, notes=False)
    ns70['diagrams'](d, notes=False)
    shutil.copy(d / 'network_early_june.png', OUT / 'fig_network_early.png')
    shutil.copy(d / 'network_july_august.png', OUT / 'fig_network_later.png')
    shutil.copy(d / 'pvo_operations.png', OUT / 'fig_pvo_system.png')
plt.rcParams.update(STYLE)
print('wrote fig_network_early, fig_network_later, fig_pvo_system')

## Figures 3 and 6: BOB coverage and RSAM–SSAM lag (copied from step 69)

In [ ]:
for src, name in [(BOB / 'overview/bob_coverage_1991.png', 'fig_bob_coverage.png'),
                  (BOB / 'overview/ssam_rsam_lag_by_day.png', 'fig_ssam_rsam_lag.png')]:
    shutil.copy(src, OUT / name)
    print('copied', src.name, '->', name)

## Figure 4: clock rules

Each row of `clock_regimes.csv`, drawn on its stream's own raw clock, coloured by the offset to UTC.

In [ ]:
t0, t1 = pd.Timestamp('1991-04-20'), pd.Timestamp('1991-12-10')
cr = pd.read_csv(PIPELINE / 'clock_regimes.csv', comment='#')
cr = pd.DataFrame([dict(stream=s, regime=r.regime, start=r.start, end=r.end, off=r.utc_offset_h)
                   for r in cr.itertuples() for s in str(r.stream).split(';')])
cr = cr[~cr.regime.str.contains('seq')]
labels = {'waveform': 'Waveform headers (H)', 'pick_individual': 'Single-event picks', 'pick_monthly': 'Monthly picks',
          'hypo_pinaall': 'PINAALL.DAT origins (W)', 'hypo_allsum': 'Pinatubo_all.sum origins (W)', 'ssam': 'SSAM (W)',
          'tandy_rsam': 'RSAM, EVNT (R)', 'ricky_lahar': 'Mudflow, rain (L)', 'cab_hourly_counts': 'Hourly CAB counts (C)',
          'sleepy_tilt': 'Tilt (T)'}
colours = {-15: '#b2182b', -8: '#ef8a62', -7: '#fddbc7', 0: '#67a9cf'}
fig, ax = plt.subplots(figsize=(7.2, 3.2))
for i, s in enumerate(labels):
    for r in cr[cr.stream == s].itertuples():
        a = max(pd.Timestamp(r.start) if isinstance(r.start, str) else t0, t0)
        b = min(pd.Timestamp(r.end) if isinstance(r.end, str) else t1, t1)
        if b <= a:
            continue
        ax.barh(i, b - a, left=a, color=colours.get(r.off, '0.8'), edgecolor='k', lw=0.3, height=0.7)
        if (b - a).days > 12:
            ax.text(a + (b - a) / 2, i, f'{r.off:+g} h', ha='center', va='center', fontsize=6)
ax.set_yticks(range(len(labels)), list(labels.values()))
ax.invert_yaxis()
for d, l in [('1991-05-26', '26 May reset'), ('1991-06-15', '15 Jun'), ('1991-09-13', '13 Sep'), ('1991-10-27', '27 Oct DST end')]:
    ax.axvline(pd.Timestamp(d), color='k', ls=':', lw=0.7)
    ax.text(pd.Timestamp(d), -0.9, l, fontsize=6, ha='center')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax.set_xlim(t0, t1)
ax.set_xlabel('Raw-clock date, 1991 (rule start/end on each stream’s own clock)')
fig.tight_layout()
save(fig, 'fig_clock_rules')

## Figure 5: effect of the clock rules

Weekly fraction of HYPO71 origins that fall inside a triggered waveform record (with 10 s of slack before the record start), on the recorded clocks and on UTC.

In [ ]:
w = pd.read_csv(TMP / '10_waveform_index.csv', usecols=['starttime', 'endtime', 'raw_starttime'], low_memory=False)
o = pd.read_csv(TMP / '43_hypo71_origin_index.csv')
o = o[o.source.isin(['hypo40', 'pinaall'])]  # HYPO71 summaries only: the later solutions use other clocks

def inside(times, starts, ends):
    s = starts.values.astype('datetime64[ns]').astype('int64')
    e = ends.values.astype('datetime64[ns]').astype('int64')
    order = np.argsort(s); s, e = s[order], e[order]
    emax = np.maximum.accumulate(e)
    t = times.values.astype('datetime64[ns]').astype('int64')
    j = np.searchsorted(s, t + 10_000_000_000, side='right') - 1
    return (j >= 0) & (emax[np.clip(j, 0, None)] >= t)

st, en = utc(w.starttime), utc(w.endtime)
rs = utc(w.raw_starttime); re_ = rs + (en - st)
ot = utc(o.origin_time)
ro = pd.to_datetime(o.raw_origin_time, utc=True, format='ISO8601', errors='coerce').dt.tz_convert(None)
ok = ro.notna()
utc_in, raw_in = inside(ot[ok], st, en), inside(ro[ok], rs, re_)
weekly = (pd.DataFrame(dict(week=ot[ok].dt.to_period('W').dt.start_time.values, utc=utc_in, raw=raw_in))
          .groupby('week').agg(utc=('utc', 'mean'), raw=('raw', 'mean')))
fig, ax = plt.subplots(figsize=(7.2, 2.4))
ax.plot(weekly.index, 100 * weekly.raw, 'o-', ms=3, color='#b2182b', label='Recorded clocks')
ax.plot(weekly.index, 100 * weekly.utc, 'o-', ms=3, color='#2166ac', label='UTC (clock rules applied)')
ax.set_ylabel('Origins inside a\nwaveform record (%)')
ax.set_ylim(0, 105)
ax.legend(frameon=False, loc='center right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax.set_xlabel('Week (UTC), 1991')
fig.tight_layout()
save(fig, 'fig_timing_effect')
print(f'origins inside a record: recorded clocks {100 * raw_in.mean():.1f}%, UTC {100 * utc_in.mean():.1f}% (n = {len(raw_in)})')

## Figure 7: clock anchors (step 72)

Four triggered records with regional or teleseismic arrivals, drawn by step 72's `plot_records` from its saved matches.

In [ ]:
ns72 = pipeline_definitions('72_external_clock_anchors.ipynb')
ANCHORS = ['9105220C', '9105301M', '91080826', '9109055G']
m = pd.read_csv(UTC / 'clock_anchor_check/waveform_matches.csv')
m['origin'] = pd.to_datetime(m.origin, utc=True)
m['p_arrival'] = pd.to_datetime(m.p_arrival, utc=True)
rows = m[m.record.isin(ANCHORS)].drop_duplicates('record').set_index('record').loc[ANCHORS].reset_index()
idx = pd.read_csv(TMP / '10_waveform_index.csv', usecols=['event_id', 'wav_file'], low_memory=False)
with plt.rc_context({'font.size': 7}):
    fig = ns72['plot_records'](rows, idx, (0.5, 3.0), (15.13, 120.35))
    fig.set_size_inches(7.2, 8.4)
    fig.tight_layout()
    save(fig, 'fig_clock_anchors', dpi=170)
rows[['record', 'origin', 'mag', 'dist_km', 'depth_km', 'predicted_p_in_record_s']]

## Figure 9: station availability (UTC days with at least one record)

In [ ]:
a = pd.read_csv(TMP / 'qc/10_station_availability_matrix.csv', index_col=0)
a.index = pd.to_datetime(a.index)
a = a.reindex(pd.date_range(a.index.min(), a.index.max(), freq='D'), fill_value=0)
order = ['UBO', 'PIE', 'PPO', 'BUR', 'GRN', 'BUG', 'CAB', 'CRW', 'DON', 'FNG', 'PI2', 'QAD']
a = a[[c for c in order if c in a.columns] + [c for c in a.columns if c not in order]]
fig, ax = plt.subplots(figsize=(7.2, 2.6))
ax.imshow(a.values.T > 0, aspect='auto', cmap='Greys', interpolation='nearest',
          extent=[mdates.date2num(a.index[0]), mdates.date2num(a.index[-1] + pd.Timedelta(days=1)), len(a.columns) - 0.5, -0.5])
ax.set_yticks(range(len(a.columns)), a.columns)
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax.axvline(mdates.date2num(pd.Timestamp('1991-06-15')), color='r', lw=0.7, ls='--')
ax.set_xlabel('Date (UTC), 1991')
fig.tight_layout()
save(fig, 'fig_station_availability')

## Figure 10: hypocenters

Preferred origins of the step 43 groups: map with latitude–depth, longitude–depth and depth–time sections. Symbol area scales with magnitude; magnitudes of 0.0 and below, and missing magnitudes, are drawn at the smallest size. Modern relief from the step 70 grid is contoured on the map (200 m interval) and drawn as east–west and north–south profiles through the summit on the sections (elevation plotted as negative depth).

In [ ]:
SUMMIT = (120.35, 15.13)
LON, LAT, DEP = (120.20, 120.50), (15.00, 15.30), (-3.0, 25.0)

def msize(m):
    m = np.where(np.isfinite(m) & (m > 0), m, 0.0)
    return 1.5 * 4.0 ** m

# Modern relief (step 70 input grid; EGM96 heights, post-1991 surface) for map contours and section profiles.
import xarray as xr
with xr.open_dataarray(Path(P['NETWORK_HISTORY_INPUT']) / 'modern_earth_relief.nc', engine='scipy') as g:
    relief = g.load()
relief_km = relief / 1000.0
lon_profile = relief_km.interp(lat=SUMMIT[1])    # east-west profile through the summit
lat_profile = relief_km.interp(lon=SUMMIT[0])    # north-south profile through the summit

e = pd.read_csv(TMP / '43_hypo71_event_index.csv').dropna(subset=['preferred_latitude', 'preferred_longitude'])
e = e.assign(t=utc(e.preferred_origin_time), s=msize(e.preferred_magnitude.to_numpy(float))).sort_values('t')
kw = dict(c=mdates.date2num(e.t), s=e.s, cmap='viridis', lw=0.15, edgecolors='k', alpha=0.7)

fig = plt.figure(figsize=(7.2, 7.0))
gs = fig.add_gridspec(2, 2, width_ratios=[2.2, 1.4], height_ratios=[2.2, 1.1], hspace=0.08, wspace=0.08,
                      left=0.10, right=0.98, top=0.95, bottom=0.16)
amap = fig.add_subplot(gs[0, 0])
alon = fig.add_subplot(gs[1, 0], sharex=amap)
alat = fig.add_subplot(gs[0, 1], sharey=amap)
atim = fig.add_subplot(gs[1, 1], sharey=alon)
for a in (amap, alon, alat, atim):
    a.spines[['top', 'right']].set_visible(True)

cs = amap.contour(relief.lon, relief.lat, relief.values, levels=np.arange(200, 1600, 200), colors='0.6',
                  linewidths=0.35, zorder=0)
amap.clabel(cs, levels=[600, 1000, 1400], fmt='%d m', fontsize=5.5, inline_spacing=1)
sc = amap.scatter(e.preferred_longitude, e.preferred_latitude, **kw)
amap.plot(*SUMMIT, '^', ms=8, mfc='none', mec='r', mew=1.2)
amap.set(xlim=LON, ylim=LAT, ylabel='Latitude (°N)')
amap.tick_params(labelbottom=False)
alon.plot(lon_profile.lon, -lon_profile.values, color='#8c510a', lw=1.0, zorder=3)
alon.scatter(e.preferred_longitude, e.preferred_depth_km, **kw)
alon.set(ylim=DEP[::-1], xlabel='Longitude (°E)', ylabel='Depth (km)')
alat.plot(-lat_profile.values, lat_profile.lat, color='#8c510a', lw=1.0, zorder=3)
alat.scatter(e.preferred_depth_km, e.preferred_latitude, **kw)
alat.set(xlim=DEP, xlabel='Depth (km)')
alat.xaxis.set_label_position('top')
alat.tick_params(labelleft=False, labelbottom=False, labeltop=True, top=True)
atim.scatter(e.t, e.preferred_depth_km, **kw)
atim.tick_params(labelleft=False)
atim.xaxis.set_major_locator(mdates.MonthLocator())
atim.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
atim.set_xlabel('Origin time (UTC), 1991')
for a in (amap, alon, alat, atim):
    a.grid(lw=0.3, alpha=0.4)
for a in (alon, atim):
    a.axhline(0, color='0.5', lw=0.5)

cb = fig.colorbar(sc, cax=fig.add_axes([0.10, 0.075, 0.43, 0.013]), orientation='horizontal')
cb.ax.xaxis.set_major_locator(mdates.MonthLocator())
cb.ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
cb.set_label('Origin time (UTC), 1991')
for mag in (0, 0.5, 1.0, 1.5):
    atim.scatter([], [], s=msize(np.array([mag]))[0], c='0.6', edgecolors='k', lw=0.2,
                 label='≤0 or none' if mag == 0 else f'{mag:g}')
fig.legend(*atim.get_legend_handles_labels(), title='Magnitude', ncol=4, frameon=False,
           loc='lower right', bbox_to_anchor=(0.98, 0.0), fontsize=7, title_fontsize=7)
save(fig, 'fig_hypocentres')

off = (~e.preferred_longitude.between(*LON) | ~e.preferred_latitude.between(*LAT)).sum()
small = (~(e.preferred_magnitude > 0)).sum()
print(f'{len(e)} groups; {off} off the map; {(e.preferred_depth_km > DEP[1]).sum()} deeper than {DEP[1]:g} km; '
      f'{small} at the smallest size (magnitude <= 0 or missing)')

## Figure 11: waveform records, picks and hypocenter groups per UTC day

In [ ]:
day = lambda s: pd.to_datetime(s, utc=True, format='ISO8601').dt.floor('D')
wf = day(w.starttime).value_counts().sort_index()
pk = day(pd.read_csv(TMP / '22_merged_pick_index.csv', usecols=['pick_time'], low_memory=False).pick_time).value_counts().sort_index()
hy = day(pd.read_csv(TMP / '43_hypo71_event_index.csv').preferred_origin_time).value_counts().sort_index()
days = pd.date_range('1991-04-28', '1991-12-07', freq='D', tz='UTC')
fig, axs = plt.subplots(3, 1, figsize=(7.2, 4.4), sharex=True)
for ax, ser, lab, c in zip(axs, [wf, pk, hy], ['Waveform records', 'Picks', 'Hypocentre groups'], ['#4d4d4d', '#1b7837', '#762a83']):
    ser = ser.reindex(days, fill_value=0)
    ax.bar(ser.index.tz_convert(None), ser.values, width=1, color=c)
    ax.set_ylabel(lab + '\nper UTC day')
    ax.axvline(pd.Timestamp('1991-06-15'), color='r', lw=0.6, ls='--')
axs[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
axs[-1].set_xlabel('Date (UTC), 1991')
fig.tight_layout()
save(fig, 'fig_daily_counts')
print('totals', wf.sum(), pk.sum(), hy.sum())

## Figure 12: 1–15 June 1991 on one UTC time axis

(a) Frequency index, log2 of the 4–20 Hz to 0.8–4 Hz mean-absolute-amplitude ratio, and (b) spectral centroid of the event windows at PIEZ and GRNZ from the analysis run, excluding windows flagged for possible clipping or with an event-to-noise ratio below 6 dB; (c) HYPO71 preferred magnitudes; (d) RSAM; (e) SSAM bands BN01–BN10 (0.5–10.5 Hz), BUR on 11 June then CAB. Eruption times are from Wolfe and Hoblitt (1996). Reading `window_metrics.csv` takes about a minute.

In [ ]:
T0, T1 = pd.Timestamp('1991-06-01'), pd.Timestamp('1991-06-16')
STATIONS = {'PIE': '#1b7837', 'GRN': '#762a83'}
SNR_DB, ROLL = 6.0, '12h'
MARKS = [(pd.Timestamp('1991-06-12 00:51'), '12 Jun'), (pd.Timestamp('1991-06-15 05:42'), '15 Jun')]

cols = ['window', 'station', 'channel', 'recorded_starttime', 'metric_flags', 'event_to_noise_rms_db',
        'sam.log2_hf_lf_ratio', 'psd.centroid_hz']
wm = pd.read_csv(ANALYSIS_RUN / 'window_metrics.csv', usecols=cols, low_memory=False)
wm = wm[(wm.window == 'event') & wm.station.isin(STATIONS) & (wm.channel.str[-1] == 'Z')]
wm = wm.assign(t=utc(wm.recorded_starttime))
wm = wm[(wm.t >= T0) & (wm.t < T1)]
n_all = len(wm)
clipped = wm.metric_flags.str.contains('clipping', na=False)
wm = wm[~clipped & (wm.event_to_noise_rms_db > SNR_DB) & np.isfinite(wm['sam.log2_hf_lf_ratio'])].sort_values('t')

hj = pd.read_csv(TMP / '43_hypo71_event_index.csv')
hj = hj.assign(t=utc(hj.preferred_origin_time))
hj = hj[(hj.t >= T0) & (hj.t < T1)]
hm = hj[hj.preferred_magnitude > 0]

r = pd.read_csv(BOB / 'rsam/rsam_records.csv', usecols=['series', 'recorded_time', 'value_01'], low_memory=False)
r = r.assign(t=pd.to_datetime(r.recorded_time, errors='coerce'))
r = r[(r.t >= T0) & (r.t < T1) & r.series.isin([s + 'Z' for s in STATIONS])]

band = [f'value_{i:02d}' for i in range(1, 11)]
s = pd.read_csv(BOB / 'ssam/ssam_records.csv', usecols=['series', 'recorded_time'] + band, low_memory=False)
s = s.assign(t=pd.to_datetime(s.recorded_time, errors='coerce'))
s = s[(s.t >= T0) & (s.t < T1)].sort_values('t')
s = pd.concat([s[s.series == 'BUR1'], s[(s.series == 'CAB1') & (s.t > s[s.series == 'BUR1'].t.max())]])

fig, ax = plt.subplots(5, 1, figsize=(7.2, 8.4), sharex=True, gridspec_kw=dict(height_ratios=[1.2, 1.2, 0.9, 0.9, 1.0]))
for key, a, lab in [('sam.log2_hf_lf_ratio', ax[0], 'Frequency index\nlog$_2$(4–20 Hz / 0.8–4 Hz)'),
                    ('psd.centroid_hz', ax[1], 'Spectral centroid (Hz)')]:
    for sta, c in STATIONS.items():
        d = wm[wm.station == sta]
        a.scatter(d.t, d[key], s=1.5, color=c, alpha=0.25, lw=0)
        med = d.set_index('t')[key].rolling(ROLL, center=True, min_periods=5).median()
        med[med.index.to_series().diff() > pd.Timedelta('6h')] = np.nan   # break the line across gaps
        a.plot(med.index, med.values, color=c, lw=1.2, label=f'{sta}Z (n = {len(d):,})')
    a.set_ylabel(lab)
ax[0].set_ylim(-3, 3)
ax[1].set_ylim(0, 20)
ax[0].legend(loc='lower left', frameon=False, ncol=2)
ax[2].scatter(hm.t, hm.preferred_magnitude, s=4, color='k', lw=0, alpha=0.6)
ax[2].set_ylabel('Magnitude\n(HYPO71)')
for sta, c in STATIONS.items():
    d = r[r.series == sta + 'Z']
    ax[3].plot(d.t, d.value_01, color=c, lw=0.7, label=f'{sta}Z')
ax[3].set_yscale('log')
ax[3].set_ylabel('RSAM\n(10 min, counts)')
ax[3].legend(loc='upper left', frameon=False, ncol=2)
z = np.log10(s[band].where(s[band] > 0).to_numpy(float)).T
tnum = mdates.date2num(s.t)
pc = ax[4].pcolormesh(np.r_[tnum, tnum[-1] + 1 / 1440], np.arange(0.5, 11.0, 1.0), z, shading='flat', cmap='magma',
                      vmin=np.nanpercentile(z, 2), vmax=np.nanpercentile(z, 99))
ax[4].set_ylabel('SSAM band (Hz)')
fig.colorbar(pc, cax=ax[4].inset_axes([1.01, 0.0, 0.012, 1.0]), label='log$_{10}$ amplitude')
for a in ax:
    for tm, _ in MARKS:
        a.axvline(tm, color='r', lw=0.6, ls='--')
for tm, lab in MARKS:
    ax[0].text(tm, 3.1, lab, color='r', ha='center', va='bottom', fontsize=7)
ax[-1].set_xlim(T0, T1)
ax[-1].xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax[-1].set_xlabel('Date (UTC), 1991')
for a, l in zip(ax, 'abcde'):
    a.text(-0.08, 1.0, f'({l})', transform=a.transAxes, fontweight='bold', va='top')
fig.align_ylabels(ax)
fig.tight_layout()
save(fig, 'fig_june_frequency')

print(f'windows {n_all} -> kept {len(wm)} ({wm.station.value_counts().to_dict()}); clipped {int(clipped.sum())}')
print(f'hypocenter groups {len(hj)}, with magnitude {len(hm)}; SSAM {s.series.unique().tolist()}')
for sta in STATIONS:
    d = wm[wm.station == sta]
    early, late = d[d.t < '1991-06-06'], d[d.t >= '1991-06-09']
    print(sta, 'FI median 1-5 Jun', round(early['sam.log2_hf_lf_ratio'].median(), 2), '9-14 Jun',
          round(late['sam.log2_hf_lf_ratio'].median(), 2), '| centroid', round(early['psd.centroid_hz'].median(), 1),
          round(late['psd.centroid_hz'].median(), 1))
rp = r[r.series == 'PIEZ'].set_index('t').value_01
print('PIEZ RSAM around 12 Jun 00:51 UTC:', rp['1991-06-12 00:30':'1991-06-12 01:10'].to_dict())

## Check

In [ ]:
import re
used = set(re.findall(r'includegraphics(?:\[[^]]*\])?\{([^}]+)\}', (PAPER / 'pinatubo_srl.tex').read_text()))
present = {p.stem for p in OUT.iterdir()}
print('figures used by the paper:', len(used))
print('missing:', sorted(used - present) or 'none')
print('in figures/ but unused:', sorted(present - used) or 'none')